# DL Practical 3: Comparative CNN Architectures (PyTorch, Colab Ready)

Run all cells top-to-bottom. Set the dataset and experiment plan in the Config cell.

In [ ]:
import os, random, math
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
seed_all(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
dataset_name = 'CIFAR10'
exp_plan = [
    ('vgg', 'Adam', 'BCE', 10),
    ('alexnet', 'SGD', 'Focal', 20),
    ('resnet18', 'Adam', 'ArcFace', 15),
]
batch_size = 128

In [ ]:
if dataset_name == 'CIFAR10':
    mean = [0.4914, 0.4822, 0.4465]; std = [0.247, 0.243, 0.261]
    transform_train = transforms.Compose([transforms.Resize(224), transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize(mean, std)])
    transform_test = transforms.Compose([transforms.Resize(224), transforms.ToTensor(), transforms.Normalize(mean, std)])
    train_dataset = datasets.CIFAR10(root='/content/data', train=True, download=True, transform=transform_train)
    test_dataset = datasets.CIFAR10(root='/content/data', train=False, download=True, transform=transform_test)
    num_classes = 10
elif dataset_name == 'MNIST':
    mean = [0.1307, 0.1307, 0.1307]; std = [0.3081, 0.3081, 0.3081]
    transform_train = transforms.Compose([transforms.Grayscale(num_output_channels=3), transforms.Resize(224), transforms.ToTensor(), transforms.Normalize(mean, std)])
    transform_test = transforms.Compose([transforms.Grayscale(num_output_channels=3), transforms.Resize(224), transforms.ToTensor(), transforms.Normalize(mean, std)])
    train_dataset = datasets.MNIST(root='/content/data', train=True, download=True, transform=transform_train)
    test_dataset = datasets.MNIST(root='/content/data', train=False, download=True, transform=transform_test)
    num_classes = 10
else:
    raise ValueError('Unsupported dataset')

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
print(dataset_name, len(train_dataset), len(test_dataset))

In [ ]:
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.5):
        super().__init__()
        self.in_features = in_features; self.out_features = out_features
        self.s = s; self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m); self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m); self.mm = math.sin(math.pi - m) * m
    def forward(self, x, labels):
        x = F.normalize(x, dim=1)
        W = F.normalize(self.weight, dim=1)
        cos = torch.mm(x, W.t())
        sin = torch.sqrt(torch.clamp(1.0 - torch.pow(cos, 2), min=0.0))
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)
        one_hot = torch.zeros_like(cos)
        one_hot.scatter_(1, labels.view(-1,1), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cos)
        output *= self.s
        return output

def arcface_infer_logits(head, feats):
    x = F.normalize(feats, dim=1)
    W = F.normalize(head.weight, dim=1)
    cos = torch.mm(x, W.t())
    return cos * head.s

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=1.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma; self.alpha = alpha; self.reduction = reduction
    def forward(self, logits, target):
        log_p = F.log_softmax(logits, dim=1)
        ll = log_p.gather(1, target.view(-1,1)).squeeze(1)
        p = torch.exp(ll)
        loss = - self.alpha * (1 - p) ** self.gamma * ll
        if self.reduction == 'mean': return loss.mean()
        if self.reduction == 'sum': return loss.sum()
        return loss

def build_model(name, num_classes):
    if name == 'vgg':
        return models.vgg11(num_classes=num_classes)
    if name == 'alexnet':
        return models.alexnet(num_classes=num_classes)
    if name == 'resnet18':
        return models.resnet18(num_classes=num_classes)
    raise ValueError('Unknown model')

def build_resnet_feature(num_classes):
    m = models.resnet18()
    in_features = m.fc.in_features
    m.fc = nn.Identity()
    return m, in_features

def one_hot(labels, num_classes):
    return F.one_hot(labels, num_classes).float()

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device, loss_name, arc_head=None, num_classes=10):
    model.train()
    total=0; correct=0; loss_sum=0.0
    for x,y in loader:
        x = x.to(device); y = y.to(device)
        optimizer.zero_grad()
        if loss_name == 'ArcFace':
            feats = model(x)
            logits = arc_head(feats, y)
            loss = criterion(logits, y)
            preds = logits.argmax(1)
        elif loss_name == 'BCE':
            logits = model(x)
            y_oh = one_hot(y, num_classes).to(logits.dtype)
            loss = F.binary_cross_entropy_with_logits(logits, y_oh)
            preds = logits.argmax(1)
        else:
            logits = model(x)
            loss = criterion(logits, y)
            preds = logits.argmax(1)
        loss.backward(); optimizer.step()
        total += x.size(0); correct += (preds == y).sum().item()
        loss_sum += loss.item() * x.size(0)
    return loss_sum/total, correct/total

def evaluate(model, loader, device, loss_name, arc_head=None):
    model.eval()
    total=0; correct=0
    with torch.no_grad():
        for x,y in loader:
            x = x.to(device); y = y.to(device)
            if loss_name == 'ArcFace':
                feats = model(x)
                logits = arcface_infer_logits(arc_head, feats)
            else:
                logits = model(x)
            preds = logits.argmax(1)
            total += x.size(0); correct += (preds == y).sum().item()
    return correct/total

In [ ]:
results = []
models_store = {}
for name, opt_name, loss_name, epochs in exp_plan:
    if loss_name == 'ArcFace':
        model, in_features = build_resnet_feature(num_classes)
        model = model.to(device)
        arc_head = ArcMarginProduct(in_features, num_classes).to(device)
        params = list(model.parameters()) + list(arc_head.parameters())
        criterion = nn.CrossEntropyLoss()
    else:
        model = build_model(name, num_classes).to(device)
        arc_head = None
        criterion = FocalLoss() if loss_name == 'Focal' else None
    if opt_name == 'Adam':
        optimizer = torch.optim.Adam(params if arc_head else model.parameters(), lr=1e-3)
    else:
        optimizer = torch.optim.SGD(params if arc_head else model.parameters(), lr=0.01, momentum=0.9)
    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion or nn.Identity(), device, loss_name, arc_head, num_classes)
    test_acc = evaluate(model, test_loader, device, loss_name, arc_head)
    results.append((name, opt_name, loss_name, epochs, train_acc, test_acc))
    models_store[f'{name}-{loss_name}'] = (model, arc_head)
for r in results:
    print(r)

In [ ]:
def collect_logits(model, head, loader, max_samples=2000, device='cpu', loss_name=None):
    xs=[]; ys=[]; count=0
    model.eval()
    with torch.no_grad():
        for x,y in loader:
            x = x.to(device)
            if loss_name == 'ArcFace':
                feats = model(x)
                logits = arcface_infer_logits(head, feats)
            else:
                logits = model(x)
            xs.append(logits.detach().cpu())
            ys.append(y)
            count += x.size(0)
            if count >= max_samples:
                break
    X = torch.cat(xs, dim=0)[:max_samples]
    Y = torch.cat(ys, dim=0)[:max_samples]
    return X.numpy(), Y.numpy()

fig, axes = plt.subplots(1, len(models_store), figsize=(5*len(models_store), 4), squeeze=False)
for idx, key in enumerate(models_store):
    model, head = models_store[key]
    loss_name = key.split('-')[-1]
    X, Y = collect_logits(model, head, test_loader, max_samples=2000, device=device, loss_name=loss_name)
    Z = TSNE(n_components=2, perplexity=30, init='random', learning_rate='auto').fit_transform(X)
    ax = axes[0][idx]
    ax.scatter(Z[:,0], Z[:,1], c=Y, s=5, cmap='tab10')
    ax.set_title(key)
    ax.set_xticks([]); ax.set_yticks([])
plt.show()